# Notebook 4 — Model Training

Train four classifiers (Logistic Regression, Decision Tree, Random Forest, XGBoost) on the preprocessed data.

## Setup & Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

from xgboost import XGBClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

## Load Preprocessed Data

In [2]:
import joblib

X_train_scaled = pd.read_csv('X_train_scaled.csv')
y_train        = pd.read_csv('y_train.csv').squeeze()

# Needed for scale_pos_weight
RANDOM_STATE = 42
print('Train shape:', X_train_scaled.shape)

Train shape: (1176, 44)


## 5. Model Development

We train four classifiers:
- Logistic Regression (baseline, interpretable)
- Decision Tree
- Random Forest
- XGBoost

`class_weight='balanced'` / `scale_pos_weight` handles the ~84/16 class imbalance.

In [3]:
# Train all 4 models
models = {}

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
models['Logistic Regression'] = log_reg

dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=RANDOM_STATE)
dt.fit(X_train_scaled, y_train)
models['Decision Tree'] = dt

rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
models['Random Forest'] = rf

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                     scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                     random_state=RANDOM_STATE, n_jobs=-1)
xgb.fit(X_train_scaled, y_train)
models['XGBoost'] = xgb

print("All 4 models trained:", list(models.keys()))

All 4 models trained: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']


## Save Trained Models

In [4]:
import joblib

for name, model in models.items():
    fname = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, fname)
    print(f'Saved {fname}')

Saved logistic_regression.pkl
Saved decision_tree.pkl
Saved random_forest.pkl
Saved xgboost.pkl
